In [ ]:
# ======================================================================
# ORIGINAL PYSPARK CODE
# ======================================================================

MAPPING FILE: mplt_CDM_ROW_WID.txt
====================================================================================================

"""
ETL Pipeline: mplt_CDM_ROW_WID
Migrated from IICS mapping: mplt_CDM_ROW_WID

Purpose:
This pipeline retrieves the maximum ROW_WID and associated TABLE_NAME from a target table using a lookup transformation,
calculates a new ROW_WID using conditional logic, and outputs the result to a Snowflake table.
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from typing import Dict
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table".
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ

    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMRowWidPipeline:
    """
    ETL Pipeline for mplt_CDM_ROW_WID.

    Sources: Custom Table
    Targets: Snowflake Table
    Transformation Logic: Lookup maximum ROW_WID, calculate new ROW_WID, and output to target.
    """

    def __init__(self, spark: SparkSession, config: Dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }

        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source.

        Optimizations:
        - Predicate pushdown for filters.
        - Column pruning to select only required fields.
        """
        logger.info("Starting data extraction")

        table_name = self.config.get('source_table', 'catalog.database.table')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(
            table_name=table_name,
            local_file_path=local_file_path
        )

        log_df_info(df, "Source: Custom Table")
        logger.info("Data extraction complete")
        return df

    def lookup_max_row_wid(self, df: DataFrame) -> DataFrame:
        """
        Perform lookup to retrieve maximum ROW_WID and associated TABLE_NAME.

        Args:
            df: Input DataFrame containing IN_TABLE_NAME.

        Returns:
            DataFrame enriched with ROW_WID and TABLE_NAME from lookup.
        """
        logger.info("Performing lookup for maximum ROW_WID")

        lookup_table_name = self.config.get('lookup_table', 'catalog.database.lookup_table')
        lookup_local_path = self.config.get('lookup_local_path', None)

        lookup_df = read_table(
            table_name=lookup_table_name,
            local_file_path=lookup_local_path
        )

        # Perform lookup using broadcast join for optimization
        enriched_df = df.join(
            F.broadcast(lookup_df.select("TABLE_NAME", "ROW_WID")),
            df["IN_TABLE_NAME"] == lookup_df["TABLE_NAME"],
            "left"
        ).fillna({"ROW_WID": 0})  # Replace null ROW_WID with 0

        log_df_info(enriched_df, "After Lookup")
        return enriched_df

    def calculate_new_row_wid(self, df: DataFrame) -> DataFrame:
        """
        Calculate new ROW_WID using conditional logic.

        Args:
            df: Input DataFrame containing ROW_WID and TABLE_NAME.

        Returns:
            DataFrame with calculated ROW_WID.
        """
        logger.info("Calculating new ROW_WID")

        df = df.withColumn(
            "V1",
            F.when(F.col("ROW_WID") == 0, F.lit(0)).otherwise(F.col("ROW_WID"))
        ).withColumn(
            "V2",
            F.col("V1") + 1
        ).withColumn(
            "ROW_WID",
            F.col("V2")
        )

        log_df_info(df, "After ROW_WID Calculation")
        return df

    def load(self, df: DataFrame):
        """
        Load data to target.

        Optimizations:
        - Delta Lake for ACID compliance.
        - Partitioning for query performance.
        """
        logger.info("Starting data load")

        target_path = self.config.get('target_path', 'path/to/target')

        log_df_info(df, "Before Load")

        (df.write
            .format("delta")
            .mode(self.config.get('write_mode', 'overwrite'))
            .partitionBy(*self.config.get('partition_columns', []))
            .option("overwriteSchema", "true")
            .save(target_path)
        )

        # Optimize target table
        if self.config.get('optimize_target', True):
            logger.info("Optimizing target table")
            zorder_cols = self.config.get('zorder_columns', [])
            if zorder_cols:
                self.spark.sql(f"""
                    OPTIMIZE delta.`{target_path}`
                    ZORDER BY ({', '.join(zorder_cols)})
                """)

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")

        try:
            # Extract
            source_df = self.extract()

            # Lookup
            lookup_df = self.lookup_max_row_wid(source_df)

            # Transform
            transformed_df = self.calculate_new_row_wid(lookup_df)

            # Load
            self.load(transformed_df)

            logger.info("ETL pipeline completed successfully")

        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Source configuration
    'source_table': 'catalog.database.custom_table',
    'source_local_path': r'path/to/custom_table.csv',

    # Target configuration
    'target_path': '/path/to/target',
    'write_mode': 'overwrite',
    'partition_columns': ['TABLE_NAME'],
    'zorder_columns': ['ROW_WID'],

    # Lookup configuration
    'lookup_table': 'catalog.database.lookup_table',
    'lookup_local_path': r'path/to/lookup_table.csv',

    # Performance configuration
    'shuffle_partitions': 200,
    'optimize_target': True
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("CDM_ROW_WID_Pipeline").getOrCreate()
    pipeline = CDMRowWidPipeline(spark, config)
    pipeline.execute()
MAPPING FILE: mplt_CDM_BATCH_ID.txt
====================================================================================================

"""
ETL Pipeline: mplt_CDM_BATCH_ID
Migrated from IICS mapping: mplt_CDM_BATCH_ID
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from typing import Dict
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table".
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ

    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMBatchIDPipeline:
    """
    ETL Pipeline for processing batch IDs.

    Sources: CDM.CDM_BATCH_CTRLID
    Targets: <output_table>
    Transformation Logic: Processes batch IDs by checking for null values, performing a lookup, and outputting either the lookup value or a default value.
    """

    def __init__(self, spark: SparkSession, config: Dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }

        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source.

        Optimizations:
        - Predicate pushdown for filters.
        - Column pruning to select only required fields.
        """
        logger.info("Starting data extraction")

        table_name = self.config.get('source_table', 'catalog.database.table')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(
            table_name=table_name,
            local_file_path=local_file_path
        )

        log_df_info(df, "Source: CDM.CDM_BATCH_CTRLID")
        logger.info("Data extraction complete")
        return df

    def transform(self, df: DataFrame) -> DataFrame:
        """
        Apply business transformations.

        Transformation Logic:
        - Pass-through SOURCE_NAME.
        - Perform lookup to retrieve maximum batch ID.
        - Check for null values and output either lookup value or default value.
        """
        logger.info("Applying transformations")

        # Step 1: Pass-through SOURCE_NAME
        df = df.select(
            F.col("SOURCE_NAME").alias("SOURCE_NAME")
        )

        # Step 2: Perform lookup to retrieve maximum batch ID
        lookup_table_name = self.config.get('lookup_table', 'catalog.database.lookup_table')
        lookup_local_path = self.config.get('lookup_local_path', None)

        lookup_df = read_table(
            table_name=lookup_table_name,
            local_file_path=lookup_local_path
        ).select(
            F.col("SOURCE_NAME").alias("lookup_SOURCE_NAME"),
            F.col("BATCH_ID").alias("lookup_BATCH_ID")
        )

        df = df.join(
            F.broadcast(lookup_df),
            df.SOURCE_NAME == lookup_df.lookup_SOURCE_NAME,
            "left"
        ).select(
            df.SOURCE_NAME,
            lookup_df.lookup_BATCH_ID.alias("LKP_BATCH_ID")
        )

        # Step 3: Check for null values and output either lookup value or default value
        df = df.withColumn(
            "o_BATCH_ID",
            F.when(F.col("LKP_BATCH_ID").isNull(), F.lit(-999)).otherwise(F.col("LKP_BATCH_ID"))
        ).select(
            F.col("o_BATCH_ID"),
            F.col("SOURCE_NAME")
        )

        log_df_info(df, "After transformations")
        return df

    def load(self, df: DataFrame):
        """
        Load data to target.

        Optimizations:
        - Delta Lake for ACID compliance.
        - Partitioning for query performance.
        """
        logger.info("Starting data load")

        target_path = self.config.get('target_path', 'path/to/target')

        log_df_info(df, "Before load")

        (df.write
            .format("delta")
            .mode(self.config.get('write_mode', 'overwrite'))
            .partitionBy(*self.config.get('partition_columns', []))
            .option("overwriteSchema", "true")
            .save(target_path)
        )

        # Optimize target table
        if self.config.get('optimize_target', True):
            logger.info("Optimizing target table")
            zorder_cols = self.config.get('zorder_columns', [])
            if zorder_cols:
                self.spark.sql(f"""
                    OPTIMIZE delta.`{target_path}`
                    ZORDER BY ({', '.join(zorder_cols)})
                """)

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")

        try:
            # Extract
            source_df = self.extract()

            # Transform
            transformed_df = self.transform(source_df)

            # Load
            self.load(transformed_df)

            logger.info("ETL pipeline completed successfully")

        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Source configuration
    'source_table': 'CDM.CDM_BATCH_CTRLID',
    'source_local_path': r'path/to/source.csv',

    # Target configuration
    'target_path': '/path/to/target',
    'write_mode': 'overwrite',
    'partition_columns': ['SOURCE_NAME'],
    'zorder_columns': ['SOURCE_NAME', 'o_BATCH_ID'],

    # Lookup configuration
    'lookup_table': 'CDM.CDM_BATCH_CTRLID',
    'lookup_local_path': r'path/to/lookup.csv',

    # Performance configuration
    'shuffle_partitions': 200,
    'optimize_target': True
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("CDM_BATCH_ID_Pipeline").getOrCreate()
    pipeline = CDMBatchIDPipeline(spark, config)
    pipeline.execute()
MAPPING FILE: m_CDM_W_CLAIM_CD_SCD3_IU.txt
====================================================================================================

"""
ETL Pipeline: m_CDM_W_CLAIM_CD_SCD3_IU
Migrated from IICS mapping: m_CDM_W_CLAIM_CD_SCD3_IU
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from typing import Dict
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table".
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ
    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMClaimSCD3Pipeline:
    """
    ETL Pipeline for processing SCD Type 3 logic for claim data.

    Sources: CDH_GW_BUR
    Targets: W_CLAIM_CD_BUR_SCD3_I (Insert), W_CLAIM_CD_BUR_SCD3_U (Update)
    Transformation Logic: SCD Type 3 handling for BUR field.
    """

    def __init__(self, spark: SparkSession, config: Dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }
        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source table CDH_GW_BUR.
        """
        logger.info("Starting data extraction")
        table_name = self.config.get('source_table', 'catalog.database.table')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(table_name=table_name, local_file_path=local_file_path)
        log_df_info(df, "Source: CDH_GW_BUR")
        return df

    def transform(self, df: DataFrame) -> (DataFrame, DataFrame):
        """
        Apply transformations including expressions, lookups, and router logic.
        """
        logger.info("Applying transformations")

        # Step 2: EXP_BUR Transformation
        df = df.select(
            F.col("POLICY_STATE").alias("INTEGRATION_ID"),
            F.col("BUR"),
            F.lit("GWCDH").alias("SOURCE_NAME")
        )
        log_df_info(df, "After EXP_BUR Transformation")

        # Step 3: Lookup Transformation
        lookup_table_name = self.config.get('lookup_table', 'catalog.database.lookup_table')
        lookup_local_path = self.config.get('lookup_local_path', None)
        lookup_df = read_table(table_name=lookup_table_name, local_file_path=lookup_local_path)

        lookup_df = lookup_df.select("LKP_ROW_WID", "LKP_INTEGRATION_ID", "LKP_NEW_BUR")
        df = df.join(
            F.broadcast(lookup_df),
            df.INTEGRATION_ID == lookup_df.LKP_INTEGRATION_ID,
            "left"
        ).select(
            df["*"],
            lookup_df["LKP_ROW_WID"],
            lookup_df["LKP_NEW_BUR"]
        )
        log_df_info(df, "After Lookup Transformation")

        # Step 4: EXP_Flag Transformation
        df = df.withColumn(
            "o_Flag",
            F.when(F.col("LKP_ROW_WID").isNull(), "I")
            .when(F.md5(F.col("BUR")) == F.md5(F.col("LKP_NEW_BUR")), "NC")
            .otherwise("U")
        ).withColumn(
            "CDM_INSERT_DT", F.current_timestamp()
        ).withColumn(
            "CDM_UPDATE_DT", F.current_timestamp()
        ).withColumn(
            "TGT_TABLE_NAME", F.lit("W_CLAIM_CD_BUR_SCD3")
        )
        log_df_info(df, "After EXP_Flag Transformation")

        # Step 5: Router Transformation
        insert_df = df.filter(F.col("o_Flag") == "I").select(
            "o_Flag", "CDM_INSERT_DT", "TGT_TABLE_NAME", "BUR"
        )
        update_df = df.filter(F.col("o_Flag") == "U").select(
            "o_Flag", "CDM_UPDATE_DT", "TGT_TABLE_NAME", "BUR"
        )
        log_df_info(insert_df, "Insert Group")
        log_df_info(update_df, "Update Group")

        return insert_df, update_df

    def load(self, insert_df: DataFrame, update_df: DataFrame):
        """
        Load data into target tables.
        """
        logger.info("Starting data load")

        # Load Insert Data
        insert_target_path = self.config.get('insert_target_path', '/path/to/insert_target')
        (insert_df.write
            .format("delta")
            .mode("append")
            .save(insert_target_path)
        )
        logger.info(f"Insert data loaded to {insert_target_path}")

        # Load Update Data
        update_target_path = self.config.get('update_target_path', '/path/to/update_target')
        (update_df.write
            .format("delta")
            .mode("append")
            .save(update_target_path)
        )
        logger.info(f"Update data loaded to {update_target_path}")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")
        try:
            # Extract
            source_df = self.extract()

            # Transform
            insert_df, update_df = self.transform(source_df)

            # Load
            self.load(insert_df, update_df)

            logger.info("ETL pipeline completed successfully")
        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise

# Configuration
config = {
    'source_table': 'CDH_GW_BUR',
    'source_local_path': r'/path/to/source.csv',
    'lookup_table': 'CDM.W_CLAIM_CD_BUR_SCD3',
    'lookup_local_path': r'/path/to/lookup.csv',
    'insert_target_path': '/path/to/insert_target',
    'update_target_path': '/path/to/update_target',
    'shuffle_partitions': 200
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("CDMClaimSCD3Pipeline").getOrCreate()
    pipeline = CDMClaimSCD3Pipeline(spark, config)
    pipeline.execute()



# ======================================================================
# AUTO-GENERATED UNIT TESTS
# Generated by PySparkUnitTestGenAgent (Chunking Strategy)
# ======================================================================

# To run: pytest <this_file>.py -v


import pytest
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F
from chispa.dataframe_comparer import assert_df_equality
from unittest.mock import Mock, patch, MagicMock, call
from decimal import Decimal


@pytest.fixture(scope="session")
def spark_session():
    """Create a SparkSession for testing"""
    spark = SparkSession.builder \
        .master("local[1]") \
        .appName("pytest-pyspark-tests") \
        .config("spark.driver.host", "localhost") \
        .config("spark.sql.shuffle.partitions", "1") \
        .getOrCreate()
    yield spark
    spark.stop()


# ======================================================================
# Tests for: Chunk 0 (Lines 1-100)
# ======================================================================

from pyspark.sql import DataFrame

def spark_session():
    return SparkSession.builder.master("local").appName("pytest").getOrCreate()

@patch("os.environ", {"DATABRICKS_RUNTIME_VERSION": "10.4"})
@patch("pyspark.sql.SparkSession.table")
def test_read_table_databricks(mock_table, spark_session):
    """
    Test read_table function when running in Databricks environment.
    """
    mock_table.return_value = spark_session.createDataFrame(
        [(1, "test_table")],
        schema=StructType([
            StructField("ROW_WID", IntegerType(), True),
            StructField("TABLE_NAME", StringType(), True)
        ])
    )
    result = read_table("catalog.database.table")
    assert result.count() == 1
    assert "ROW_WID" in result.columns
    assert "TABLE_NAME" in result.columns

@patch("os.environ", {})
@patch("pyspark.sql.SparkSession.read.csv")
def test_read_table_local_file(mock_csv, spark_session):
    """
    Test read_table function when running in local environment with valid file path.
    """
    mock_csv.return_value = spark_session.createDataFrame(
        [(1, "test_table")],
        schema=StructType([
            StructField("ROW_WID", IntegerType(), True),
            StructField("TABLE_NAME", StringType(), True)
        ])
    )
    result = read_table("catalog.database.table", local_file_path="test.csv")
    assert result.count() == 1
    assert "ROW_WID" in result.columns
    assert "TABLE_NAME" in result.columns

@patch("os.environ", {})
def test_read_table_local_file_missing_path(spark_session):
    """
    Test read_table function when running in local environment without file path.
    """
    with pytest.raises(Exception, match="Local file path required for catalog.database.table"):
        read_table("catalog.database.table")

def test_log_df_info_valid_dataframe(spark_session):
    """
    Test log_df_info function with a valid DataFrame.
    """
    df = spark_session.createDataFrame(
        [(1, "test_table")],
        schema=StructType([
            StructField("ROW_WID", IntegerType(), True),
            StructField("TABLE_NAME", StringType(), True)
        ])
    )
    with patch("logging.Logger.info") as mock_info:
        log_df_info(df, "Test Step")
        mock_info.assert_any_call("Test Step: 2 columns")
        mock_info.assert_any_call("Schema: ['ROW_WID:IntegerType', 'TABLE_NAME:StringType']")

def test_log_df_info_empty_dataframe(spark_session):
    """
    Test log_df_info function with an empty DataFrame.
    """
    df = spark_session.createDataFrame([], StructType([]))
    with patch("logging.Logger.info") as mock_info:
        log_df_info(df, "Empty Step")
        mock_info.assert_any_call("Empty Step: 0 columns")
        mock_info.assert_any_call("Schema: []")

def test_log_df_info_error_handling(spark_session):
    """
    Test log_df_info function when an error occurs during logging.
    """
    df = None  # Simulate an invalid DataFrame
    with patch("logging.Logger.error") as mock_error:
        log_df_info(df, "Error Step")
        mock_error.assert_called_once_with("Error logging DataFrame info: 'NoneType' object has no attribute 'columns'")

def test_CDMRowWidPipeline_initialization(spark_session):
    """
    Test initialization of CDMRowWidPipeline class.
    """
    config = {"shuffle_partitions": 100}
    pipeline = CDMRowWidPipeline(spark_session, config)
    assert pipeline.spark == spark_session
    assert pipeline.config == config

def test_CDMRowWidPipeline_configure_spark(spark_session):
    """
    Test _configure_spark method of CDMRowWidPipeline class.
    """
    config = {"shuffle_partitions": 100}
    pipeline = CDMRowWidPipeline(spark_session, config)
    pipeline._configure_spark()
    assert spark_session.conf.get("spark.sql.adaptive.enabled") == "true"
    assert spark_session.conf.get("spark.sql.adaptive.coalescePartitions.enabled") == "true"
    assert spark_session.conf.get("spark.sql.shuffle.partitions") == "100"

@patch("logging.Logger.info")
def test_CDMRowWidPipeline_extract_logging(mock_info, spark_session):
    """
    Test extract method logging in CDMRowWidPipeline class.
    """
    config = {"source_table": "catalog.database.table"}
    pipeline = CDMRowWidPipeline(spark_session, config)
    pipeline.extract()
    mock_info.assert_called_once_with("Starting data extraction")

# ======================================================================
# Tests for: Chunk 1 (Lines 81-180)
# ======================================================================

def spark_session():
    return SparkSession.builder.master("local").appName("pytest").getOrCreate()

@patch("module_name.read_table")
def test_extract_valid_data(mock_read_table, spark_session):
    """
    Test extract function with valid data.
    """
    # Mocking read_table
    test_data = [("table1", 1), ("table2", 2)]
    schema = StructType([
        StructField("IN_TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    mock_df = spark_session.createDataFrame(test_data, schema)
    mock_read_table.return_value = mock_df

    # Configurations
    config = {"source_table": "catalog.database.table", "source_local_path": None}
    instance = YourClass(spark_session, config)

    # Execute function
    result_df = instance.extract()

    # Validate results
    assert_df_equality(result_df, mock_df)

@patch("module_name.read_table")
def test_lookup_max_row_wid_valid_data(mock_read_table, spark_session):
    """
    Test lookup_max_row_wid function with valid data.
    """
    # Mocking read_table
    lookup_data = [("table1", 100), ("table2", 200)]
    lookup_schema = StructType([
        StructField("TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    mock_lookup_df = spark_session.createDataFrame(lookup_data, lookup_schema)
    mock_read_table.return_value = mock_lookup_df

    # Input DataFrame
    input_data = [("table1",), ("table2",)]
    input_schema = StructType([
        StructField("IN_TABLE_NAME", StringType(), True)
    ])
    input_df = spark_session.createDataFrame(input_data, input_schema)

    # Configurations
    config = {"lookup_table": "catalog.database.lookup_table", "lookup_local_path": None}
    instance = YourClass(spark_session, config)

    # Execute function
    result_df = instance.lookup_max_row_wid(input_df)

    # Expected DataFrame
    expected_data = [("table1", 100), ("table2", 200)]
    expected_schema = StructType([
        StructField("IN_TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    expected_df = spark_session.createDataFrame(expected_data, expected_schema)

    # Validate results
    assert_df_equality(result_df, expected_df)

def test_calculate_new_row_wid_valid_data(spark_session):
    """
    Test calculate_new_row_wid function with valid data.
    """
    # Input DataFrame
    input_data = [("table1", 0), ("table2", 100)]
    input_schema = StructType([
        StructField("TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    input_df = spark_session.createDataFrame(input_data, input_schema)

    # Configurations
    config = {}
    instance = YourClass(spark_session, config)

    # Execute function
    result_df = instance.calculate_new_row_wid(input_df)

    # Expected DataFrame
    expected_data = [("table1", 1), ("table2", 101)]
    expected_schema = StructType([
        StructField("TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    expected_df = spark_session.createDataFrame(expected_data, expected_schema)

    # Validate results
    assert_df_equality(result_df, expected_df)

@patch("module_name.log_df_info")
@patch("module_name.read_table")
def test_load_valid_data(mock_read_table, mock_log_df_info, spark_session):
    """
    Test load function with valid data.
    """
    # Input DataFrame
    input_data = [("table1", 1), ("table2", 2)]
    input_schema = StructType([
        StructField("TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    input_df = spark_session.createDataFrame(input_data, input_schema)

    # Mocking log_df_info
    mock_log_df_info.return_value = None

    # Configurations
    config = {"target_path": "path/to/target"}
    instance = YourClass(spark_session, config)

    # Mocking write operation
    with patch.object(input_df.write, "format") as mock_write_format:
        mock_write = MagicMock()
        mock_write_format.return_value = mock_write
        mock_write.save.return_value = None

        # Execute function
        instance.load(input_df)

        # Validate write operation
        mock_write_format.assert_called_once_with("delta")
        mock_write.save.assert_called_once_with("path/to/target")

# ======================================================================
# Tests for: Chunk 2 (Lines 161-260)
# ======================================================================

def spark_session():
    return SparkSession.builder.master("local").appName("pytest").getOrCreate()

@patch("pyspark.sql.DataFrame.write")
def test_load_happy_path(mock_write, spark_session):
    """
    Test the load method with valid input DataFrame and configuration.
    """
    # Mock configuration
    config = {
        'target_path': '/path/to/target',
        'write_mode': 'overwrite',
        'partition_columns': ['TABLE_NAME'],
        'optimize_target': True,
        'zorder_columns': ['ROW_WID']
    }

    # Create test DataFrame
    test_data = [("table1", 1), ("table2", 2)]
    schema = StructType([
        StructField("TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    df = spark_session.createDataFrame(test_data, schema)

    # Mock write operation
    mock_write.format.return_value.mode.return_value.partitionBy.return_value.option.return_value.save.return_value = None

    # Mock SQL execution for optimization
    with patch.object(spark_session, "sql") as mock_sql:
        pipeline = CDMRowWidPipeline(spark_session, config)
        pipeline.load(df)

        # Assertions
        mock_write.format.assert_called_with("delta")
        mock_write.format.return_value.mode.assert_called_with("overwrite")
        mock_write.format.return_value.mode.return_value.partitionBy.assert_called_with("TABLE_NAME")
        mock_write.format.return_value.mode.return_value.partitionBy.return_value.option.assert_called_with("overwriteSchema", "true")
        mock_write.format.return_value.mode.return_value.partitionBy.return_value.option.return_value.save.assert_called_with('/path/to/target')
        mock_sql.assert_called_once_with("OPTIMIZE delta.`/path/to/target` ZORDER BY (ROW_WID)")

@patch("pyspark.sql.DataFrame.write")
def test_load_no_partition(mock_write, spark_session):
    """
    Test the load method when no partition columns are provided.
    """
    # Mock configuration
    config = {
        'target_path': '/path/to/target',
        'write_mode': 'overwrite',
        'partition_columns': [],
        'optimize_target': False
    }

    # Create test DataFrame
    test_data = [("table1", 1), ("table2", 2)]
    schema = StructType([
        StructField("TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    df = spark_session.createDataFrame(test_data, schema)

    # Mock write operation
    mock_write.format.return_value.mode.return_value.partitionBy.return_value.option.return_value.save.return_value = None

    pipeline = CDMRowWidPipeline(spark_session, config)
    pipeline.load(df)

    # Assertions
    mock_write.format.assert_called_with("delta")
    mock_write.format.return_value.mode.assert_called_with("overwrite")
    mock_write.format.return_value.mode.return_value.partitionBy.assert_called_with()
    mock_write.format.return_value.mode.return_value.partitionBy.return_value.option.assert_called_with("overwriteSchema", "true")
    mock_write.format.return_value.mode.return_value.partitionBy.return_value.option.return_value.save.assert_called_with('/path/to/target')

@patch("pyspark.sql.DataFrame.write")
def test_load_invalid_path(mock_write, spark_session):
    """
    Test the load method with an invalid target path.
    """
    # Mock configuration
    config = {
        'target_path': None,
        'write_mode': 'overwrite',
        'partition_columns': ['TABLE_NAME'],
        'optimize_target': True
    }

    # Create test DataFrame
    test_data = [("table1", 1), ("table2", 2)]
    schema = StructType([
        StructField("TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    df = spark_session.createDataFrame(test_data, schema)

    pipeline = CDMRowWidPipeline(spark_session, config)

    with pytest.raises(Exception, match="Target path is not defined"):
        pipeline.load(df)

@patch("pyspark.sql.DataFrame.write")
def test_load_no_optimization(mock_write, spark_session):
    """
    Test the load method when optimization is disabled.
    """
    # Mock configuration
    config = {
        'target_path': '/path/to/target',
        'write_mode': 'overwrite',
        'partition_columns': ['TABLE_NAME'],
        'optimize_target': False
    }

    # Create test DataFrame
    test_data = [("table1", 1), ("table2", 2)]
    schema = StructType([
        StructField("TABLE_NAME", StringType(), True),
        StructField("ROW_WID", IntegerType(), True)
    ])
    df = spark_session.createDataFrame(test_data, schema)

    # Mock write operation
    mock_write.format.return_value.mode.return_value.partitionBy.return_value.option.return_value.save.return_value = None

    pipeline = CDMRowWidPipeline(spark_session, config)
    pipeline.load(df)

    # Assertions
    mock_write.format.assert_called_with("delta")
    mock_write.format.return_value.mode.assert_called_with("overwrite")
    mock_write.format.return_value.mode.return_value.partitionBy.assert_called_with("TABLE_NAME")
    mock_write.format.return_value.mode.return_value.partitionBy.return_value.option.assert_called_with("overwriteSchema", "true")
    mock_write.format.return_value.mode.return_value.partitionBy.return_value.option.return_value.save.assert_called_with('/path/to/target')

def test_execute_happy_path(spark_session):
    """
    Test the execute method for a successful ETL pipeline run.
    """
    # Mock configuration
    config = {
        'source_table': 'catalog.database.custom_table',
        'target_path': '/path/to/target',
        'write_mode': 'overwrite',
        'partition_columns': ['TABLE_NAME'],
        'optimize_target': True,
        'zorder_columns': ['ROW_WID']
    }

    # Mock pipeline methods
    with patch.object(CDMRowWidPipeline, "extract") as mock_extract, \
         patch.object(CDMRowWidPipeline, "lookup_max_row_wid") as mock_lookup, \
         patch.object(CDMRowWidPipeline, "calculate_new_row_wid") as mock_transform, \
         patch.object(CDMRowWidPipeline, "load") as mock_load:

        mock_extract.return_value = MagicMock()
        mock_lookup.return_value = MagicMock()
        mock_transform.return_value = MagicMock()

        pipeline = CDMRowWidPipeline(spark_session, config)
        pipeline.execute()

        # Assertions
        mock_extract.assert_called_once()
        mock_lookup.assert_called_once_with(mock_extract.return_value)
        mock_transform.assert_called_once_with(mock_lookup.return_value)
        mock_load.assert_called_once_with(mock_transform.return_value)

# ======================================================================
# Tests for: Chunk 3 (Lines 241-340)
# ======================================================================

from pyspark.sql import DataFrame

def spark_session():
    return SparkSession.builder.master("local").appName("pytest").getOrCreate()

@patch("os.environ", {"DATABRICKS_RUNTIME_VERSION": "10.4"})
@patch("pyspark.sql.SparkSession.table")
def test_read_table_databricks(mock_table, spark_session):
    """
    Test read_table function when running in Databricks environment.
    """
    mock_table.return_value = spark_session.createDataFrame(
        [(1, "value1"), (2, "value2")],
        StructType([
            StructField("id", IntegerType(), True),
            StructField("value", StringType(), True)
        ])
    )
    result = read_table("catalog.database.table")
    assert result.count() == 2
    assert "id" in result.columns
    assert "value" in result.columns

@patch("os.environ", {})
@patch("pyspark.sql.SparkSession.read")
def test_read_table_local_file(mock_read, spark_session):
    """
    Test read_table function when running in local environment with valid file path.
    """
    mock_read.csv.return_value = spark_session.createDataFrame(
        [(1, "value1"), (2, "value2")],
        StructType([
            StructField("id", IntegerType(), True),
            StructField("value", StringType(), True)
        ])
    )
    result = read_table("catalog.database.table", "path/to/file.csv")
    assert result.count() == 2
    assert "id" in result.columns
    assert "value" in result.columns

@patch("os.environ", {})
def test_read_table_local_file_missing_path(spark_session):
    """
    Test read_table function when running in local environment without file path.
    """
    with pytest.raises(Exception, match="Local file path required for catalog.database.table"):
        read_table("catalog.database.table")

def test_log_df_info_valid_dataframe(spark_session, caplog):
    """
    Test log_df_info function with a valid DataFrame.
    """
    df = spark_session.createDataFrame(
        [(1, "value1"), (2, "value2")],
        StructType([
            StructField("id", IntegerType(), True),
            StructField("value", StringType(), True)
        ])
    )
    log_df_info(df, "Test Step")
    assert "Test Step: 2 columns" in caplog.text
    assert "Schema: ['id:IntegerType', 'value:StringType']" in caplog.text

def test_log_df_info_empty_dataframe(spark_session, caplog):
    """
    Test log_df_info function with an empty DataFrame.
    """
    df = spark_session.createDataFrame([], StructType([]))
    log_df_info(df, "Empty Step")
    assert "Empty Step: 0 columns" in caplog.text
    assert "Schema: []" in caplog.text

def test_log_df_info_error_handling(spark_session, caplog):
    """
    Test log_df_info function when an error occurs during logging.
    """
    df = None  # Invalid DataFrame
    log_df_info(df, "Error Step")
    assert "Error logging DataFrame info" in caplog.text

def test_CDMBatchIDPipeline_init(spark_session):
    """
    Test initialization of CDMBatchIDPipeline class.
    """
    config = {"shuffle_partitions": 100}
    pipeline = CDMBatchIDPipeline(spark_session, config)
    assert pipeline.spark == spark_session
    assert pipeline.config == config

def test_CDMBatchIDPipeline_configure_spark(spark_session):
    """
    Test _configure_spark method of CDMBatchIDPipeline class.
    """
    config = {"shuffle_partitions": 100}
    pipeline = CDMBatchIDPipeline(spark_session, config)
    pipeline._configure_spark()
    assert spark_session.conf.get("spark.sql.adaptive.enabled") == "true"
    assert spark_session.conf.get("spark.sql.adaptive.coalescePartitions.enabled") == "true"
    assert spark_session.conf.get("spark.sql.shuffle.partitions") == "100"

# ======================================================================
# Tests for: Chunk 4 (Lines 321-420)
# ======================================================================

def spark_session():
    return SparkSession.builder.master("local").appName("pytest").getOrCreate()

@patch("module_under_test.read_table")
def test_extract_valid_data(mock_read_table, spark_session):
    """
    Test extract method with valid data from source table.
    """
    # Mock configuration
    config = {
        'source_table': 'catalog.database.table',
        'source_local_path': None
    }

    # Mock data
    test_data = [("source1", 1), ("source2", 2)]
    schema = StructType([
        StructField("SOURCE_NAME", StringType(), True),
        StructField("BATCH_ID", IntegerType(), True)
    ])
    mock_df = spark_session.createDataFrame(test_data, schema)
    mock_read_table.return_value = mock_df

    # Initialize object and call extract
    etl = module_under_test.ETL(spark_session, config)
    result_df = etl.extract()

    # Assertions
    assert_df_equality(result_df, mock_df)

@patch("module_under_test.read_table")
def test_extract_empty_data(mock_read_table, spark_session):
    """
    Test extract method with empty data from source table.
    """
    # Mock configuration
    config = {
        'source_table': 'catalog.database.table',
        'source_local_path': None
    }

    # Mock empty data
    schema = StructType([
        StructField("SOURCE_NAME", StringType(), True),
        StructField("BATCH_ID", IntegerType(), True)
    ])
    mock_df = spark_session.createDataFrame([], schema)
    mock_read_table.return_value = mock_df

    # Initialize object and call extract
    etl = module_under_test.ETL(spark_session, config)
    result_df = etl.extract()

    # Assertions
    assert result_df.count() == 0

@patch("module_under_test.read_table")
def test_transform_valid_data(mock_read_table, spark_session):
    """
    Test transform method with valid input data.
    """
    # Mock configuration
    config = {
        'lookup_table': 'catalog.database.lookup_table',
        'lookup_local_path': None
    }

    # Mock input data
    input_data = [("source1",), ("source2",)]
    input_schema = StructType([
        StructField("SOURCE_NAME", StringType(), True)
    ])
    input_df = spark_session.createDataFrame(input_data, input_schema)

    # Mock lookup data
    lookup_data = [("source1", 100), ("source2", 200)]
    lookup_schema = StructType([
        StructField("lookup_SOURCE_NAME", StringType(), True),
        StructField("lookup_BATCH_ID", IntegerType(), True)
    ])
    lookup_df = spark_session.createDataFrame(lookup_data, lookup_schema)
    mock_read_table.return_value = lookup_df

    # Initialize object and call transform
    etl = module_under_test.ETL(spark_session, config)
    result_df = etl.transform(input_df)

    # Expected output
    expected_data = [(100, "source1"), (200, "source2")]
    expected_schema = StructType([
        StructField("o_BATCH_ID", IntegerType(), True),
        StructField("SOURCE_NAME", StringType(), True)
    ])
    expected_df = spark_session.createDataFrame(expected_data, expected_schema)

    # Assertions
    assert_df_equality(result_df, expected_df)

@patch("module_under_test.read_table")
def test_transform_null_lookup(mock_read_table, spark_session):
    """
    Test transform method with null lookup values.
    """
    # Mock configuration
    config = {
        'lookup_table': 'catalog.database.lookup_table',
        'lookup_local_path': None
    }

    # Mock input data
    input_data = [("source1",), ("source3",)]
    input_schema = StructType([
        StructField("SOURCE_NAME", StringType(), True)
    ])
    input_df = spark_session.createDataFrame(input_data, input_schema)

    # Mock lookup data
    lookup_data = [("source1", 100)]
    lookup_schema = StructType([
        StructField("lookup_SOURCE_NAME", StringType(), True),
        StructField("lookup_BATCH_ID", IntegerType(), True)
    ])
    lookup_df = spark_session.createDataFrame(lookup_data, lookup_schema)
    mock_read_table.return_value = lookup_df

    # Initialize object and call transform
    etl = module_under_test.ETL(spark_session, config)
    result_df = etl.transform(input_df)

    # Expected output
    expected_data = [(100, "source1"), (-999, "source3")]
    expected_schema = StructType([
        StructField("o_BATCH_ID", IntegerType(), True),
        StructField("SOURCE_NAME", StringType(), True)
    ])
    expected_df = spark_session.createDataFrame(expected_data, expected_schema)

    # Assertions
    assert_df_equality(result_df, expected_df)

@patch("module_under_test.read_table")
def test_transform_empty_input(mock_read_table, spark_session):
    """
    Test transform method with empty input DataFrame.
    """
    # Mock configuration
    config = {
        'lookup_table': 'catalog.database.lookup_table',
        'lookup_local_path': None
    }

    # Mock empty input data
    input_schema = StructType([
        StructField("SOURCE_NAME", StringType(), True)
    ])
    input_df = spark_session.createDataFrame([], input_schema)

    # Mock lookup data
    lookup_data = [("source1", 100)]
    lookup_schema = StructType([
        StructField("lookup_SOURCE_NAME", StringType(), True),
        StructField("lookup_BATCH_ID", IntegerType(), True)
    ])
    lookup_df = spark_session.createDataFrame(lookup_data, lookup_schema)
    mock_read_table.return_value = lookup_df

    # Initialize object and call transform
    etl = module_under_test.ETL(spark_session, config)
    result_df = etl.transform(input_df)

    # Assertions
    assert result_df.count() == 0

# ======================================================================
# Tests for: Chunk 5 (Lines 401-500)
# ======================================================================

def spark_session():
    return SparkSession.builder.master("local").appName("pytest").getOrCreate()

@patch("CDMBatchIDPipeline.extract")
def test_extract_valid_data(mock_extract, spark_session):
    """
    Test the extract method with valid data.
    """
    # Mock the extract method
    test_data = [("1", "source1"), ("2", "source2")]
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    mock_df = spark_session.createDataFrame(test_data, schema)
    mock_extract.return_value = mock_df

    pipeline = CDMBatchIDPipeline(spark_session, {})
    result_df = pipeline.extract()

    assert_df_equality(result_df, mock_df)

@patch("CDMBatchIDPipeline.extract")
def test_extract_empty_data(mock_extract, spark_session):
    """
    Test the extract method with empty data.
    """
    # Mock the extract method
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    mock_df = spark_session.createDataFrame([], schema)
    mock_extract.return_value = mock_df

    pipeline = CDMBatchIDPipeline(spark_session, {})
    result_df = pipeline.extract()

    assert result_df.count() == 0

@patch("CDMBatchIDPipeline.extract")
def test_extract_invalid_schema(mock_extract, spark_session):
    """
    Test the extract method with invalid schema.
    """
    # Mock the extract method
    test_data = [("1", "source1"), ("2", "source2")]
    invalid_schema = StructType([
        StructField("invalid_field", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    mock_df = spark_session.createDataFrame(test_data, invalid_schema)
    mock_extract.return_value = mock_df

    pipeline = CDMBatchIDPipeline(spark_session, {})
    result_df = pipeline.extract()

    assert "id" not in result_df.columns

def test_transform_valid_data(spark_session):
    """
    Test the transform method with valid data.
    """
    pipeline = CDMBatchIDPipeline(spark_session, {})
    test_data = [("1", "source1"), ("2", "source2")]
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    input_df = spark_session.createDataFrame(test_data, schema)

    result_df = pipeline.transform(input_df)

    assert result_df.count() == 2
    assert "o_BATCH_ID" in result_df.columns

def test_transform_empty_data(spark_session):
    """
    Test the transform method with empty data.
    """
    pipeline = CDMBatchIDPipeline(spark_session, {})
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    input_df = spark_session.createDataFrame([], schema)

    result_df = pipeline.transform(input_df)

    assert result_df.count() == 0

@patch("pyspark.sql.DataFrame.write")
def test_load_valid_data(mock_write, spark_session):
    """
    Test the load method with valid data.
    """
    pipeline = CDMBatchIDPipeline(spark_session, {"target_path": "/path/to/target"})
    test_data = [("1", "source1"), ("2", "source2")]
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    input_df = spark_session.createDataFrame(test_data, schema)

    pipeline.load(input_df)

    mock_write.format.assert_called_with("delta")
    mock_write.mode.assert_called_with("overwrite")
    mock_write.save.assert_called_with("/path/to/target")

@patch("pyspark.sql.DataFrame.write")
def test_load_empty_data(mock_write, spark_session):
    """
    Test the load method with empty data.
    """
    pipeline = CDMBatchIDPipeline(spark_session, {"target_path": "/path/to/target"})
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    input_df = spark_session.createDataFrame([], schema)

    pipeline.load(input_df)

    mock_write.format.assert_called_with("delta")
    mock_write.mode.assert_called_with("overwrite")
    mock_write.save.assert_called_with("/path/to/target")

@patch("CDMBatchIDPipeline.extract")
@patch("CDMBatchIDPipeline.transform")
@patch("CDMBatchIDPipeline.load")
def test_execute_success(mock_load, mock_transform, mock_extract, spark_session):
    """
    Test the execute method for successful pipeline execution.
    """
    pipeline = CDMBatchIDPipeline(spark_session, {})
    test_data = [("1", "source1"), ("2", "source2")]
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    mock_df = spark_session.createDataFrame(test_data, schema)
    mock_extract.return_value = mock_df
    mock_transform.return_value = mock_df

    pipeline.execute()

    mock_extract.assert_called_once()
    mock_transform.assert_called_once_with(mock_df)
    mock_load.assert_called_once_with(mock_df)

@patch("CDMBatchIDPipeline.extract", side_effect=Exception("Extract failed"))
def test_execute_extract_failure(mock_extract, spark_session):
    """
    Test the execute method when extract fails.
    """
    pipeline = CDMBatchIDPipeline(spark_session, {})

    with pytest.raises(Exception, match="Extract failed"):
        pipeline.execute()

@patch("CDMBatchIDPipeline.transform", side_effect=Exception("Transform failed"))
def test_execute_transform_failure(mock_transform, spark_session):
    """
    Test the execute method when transform fails.
    """
    pipeline = CDMBatchIDPipeline(spark_session, {})
    test_data = [("1", "source1"), ("2", "source2")]
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    mock_df = spark_session.createDataFrame(test_data, schema)

    with patch("CDMBatchIDPipeline.extract", return_value=mock_df):
        with pytest.raises(Exception, match="Transform failed"):
            pipeline.execute()

@patch("CDMBatchIDPipeline.load", side_effect=Exception("Load failed"))
def test_execute_load_failure(mock_load, spark_session):
    """
    Test the execute method when load fails.
    """
    pipeline = CDMBatchIDPipeline(spark_session, {})
    test_data = [("1", "source1"), ("2", "source2")]
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("source_name", StringType(), True)
    ])
    mock_df = spark_session.createDataFrame(test_data, schema)

    with patch("CDMBatchIDPipeline.extract", return_value=mock_df):
        with patch("CDMBatchIDPipeline.transform", return_value=mock_df):
            with pytest.raises(Exception, match="Load failed"):
                pipeline.execute()

# ======================================================================
# Tests for: Chunk 6 (Lines 481-580)
# ======================================================================

from pyspark.sql import DataFrame

def spark_session():
    return SparkSession.builder.master("local").appName("pytest").getOrCreate()

@patch("os.environ", {"DATABRICKS_RUNTIME_VERSION": "10.4"})
@patch("pyspark.sql.SparkSession.table")
def test_read_table_databricks(mock_table, spark_session):
    """
    Test read_table function when running in Databricks environment.
    """
    mock_table.return_value = spark_session.createDataFrame(
        [(1, "Alice"), (2, "Bob")],
        StructType([
            StructField("id", IntegerType(), True),
            StructField("name", StringType(), True)
        ])
    )
    result = read_table("catalog.database.table")
    assert result.count() == 2
    assert "id" in result.columns
    assert "name" in result.columns

@patch("os.environ", {})
@patch("pyspark.sql.DataFrameReader.csv")
def test_read_table_local_file(mock_csv, spark_session):
    """
    Test read_table function when running in local environment with valid file path.
    """
    mock_csv.return_value = spark_session.createDataFrame(
        [(1, "Alice"), (2, "Bob")],
        StructType([
            StructField("id", IntegerType(), True),
            StructField("name", StringType(), True)
        ])
    )
    result = read_table("catalog.database.table", "path/to/file.csv")
    assert result.count() == 2
    assert "id" in result.columns
    assert "name" in result.columns

@patch("os.environ", {})
def test_read_table_local_file_missing_path(spark_session):
    """
    Test read_table function when running in local environment without file path.
    """
    with pytest.raises(Exception, match="Local file path required for catalog.database.table"):
        read_table("catalog.database.table")

@patch("pyspark.sql.DataFrame.schema")
def test_log_df_info_valid_dataframe(mock_schema, spark_session):
    """
    Test log_df_info function with a valid DataFrame.
    """
    mock_schema.fields = [
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True)
    ]
    df = spark_session.createDataFrame(
        [(1, "Alice"), (2, "Bob")],
        StructType([
            StructField("id", IntegerType(), True),
            StructField("name", StringType(), True)
        ])
    )
    with patch("logging.Logger.info") as mock_info:
        log_df_info(df, "Test Step")
        mock_info.assert_any_call("Test Step: 2 columns")
        mock_info.assert_any_call("Schema: ['id:IntegerType', 'name:StringType']")

def test_log_df_info_empty_dataframe(spark_session):
    """
    Test log_df_info function with an empty DataFrame.
    """
    df = spark_session.createDataFrame([], StructType([]))
    with patch("logging.Logger.info") as mock_info:
        log_df_info(df, "Empty Step")
        mock_info.assert_any_call("Empty Step: 0 columns")
        mock_info.assert_any_call("Schema: []")

def test_CDMClaimSCD3Pipeline_init(spark_session):
    """
    Test initialization of CDMClaimSCD3Pipeline class.
    """
    config = {"shuffle_partitions": 100}
    pipeline = CDMClaimSCD3Pipeline(spark_session, config)
    assert pipeline.spark == spark_session
    assert pipeline.config == config

def test_CDMClaimSCD3Pipeline_configure_spark(spark_session):
    """
    Test _configure_spark method of CDMClaimSCD3Pipeline class.
    """
    config = {"shuffle_partitions": 100}
    pipeline = CDMClaimSCD3Pipeline(spark_session, config)
    pipeline._configure_spark()
    assert spark_session.conf.get("spark.sql.adaptive.enabled") == "true"
    assert spark_session.conf.get("spark.sql.adaptive.coalescePartitions.enabled") == "true"
    assert spark_session.conf.get("spark.sql.shuffle.partitions") == "100"

# ======================================================================
# Tests for: Chunk 7 (Lines 561-660)
# ======================================================================

def spark_session():
    return SparkSession.builder.master("local").appName("pytest").getOrCreate()

@patch("module_under_test.read_table")
@patch("module_under_test.log_df_info")
def test_extract_valid_table(mock_log_df_info, mock_read_table, spark_session):
    """
    Test extract method with valid table configuration.
    """
    mock_read_table.return_value = spark_session.createDataFrame(
        [("CA", "BUR1"), ("NY", "BUR2")],
        StructType([
            StructField("POLICY_STATE", StringType(), True),
            StructField("BUR", StringType(), True)
        ])
    )
    config = {"source_table": "catalog.database.table"}
    instance = module_under_test(spark_session, config)
    result_df = instance.extract()

    assert result_df.count() == 2
    assert "POLICY_STATE" in result_df.columns
    assert "BUR" in result_df.columns
    mock_log_df_info.assert_called_once()

@patch("module_under_test.read_table")
@patch("module_under_test.log_df_info")
def test_extract_missing_table(mock_log_df_info, mock_read_table, spark_session):
    """
    Test extract method when table is missing.
    """
    mock_read_table.side_effect = Exception("Table not found")
    config = {"source_table": "catalog.database.missing_table"}
    instance = module_under_test(spark_session, config)

    with pytest.raises(Exception, match="Table not found"):
        instance.extract()

@patch("module_under_test.read_table")
@patch("module_under_test.log_df_info")
def test_transform_valid_data(mock_log_df_info, mock_read_table, spark_session):
    """
    Test transform method with valid input data.
    """
    source_df = spark_session.createDataFrame(
        [("CA", "BUR1"), ("NY", "BUR2")],
        StructType([
            StructField("POLICY_STATE", StringType(), True),
            StructField("BUR", StringType(), True)
        ])
    )
    lookup_df = spark_session.createDataFrame(
        [("ROW1", "CA", "BUR1"), ("ROW2", "NY", "BUR3")],
        StructType([
            StructField("LKP_ROW_WID", StringType(), True),
            StructField("LKP_INTEGRATION_ID", StringType(), True),
            StructField("LKP_NEW_BUR", StringType(), True)
        ])
    )
    mock_read_table.side_effect = [lookup_df]
    config = {"lookup_table": "catalog.database.lookup_table"}
    instance = module_under_test(spark_session, config)

    insert_df, update_df = instance.transform(source_df)

    assert insert_df.count() == 1
    assert update_df.count() == 1
    assert "o_Flag" in insert_df.columns
    assert "o_Flag" in update_df.columns
    mock_log_df_info.assert_called()

@patch("module_under_test.read_table")
@patch("module_under_test.log_df_info")
def test_transform_empty_data(mock_log_df_info, mock_read_table, spark_session):
    """
    Test transform method with empty input data.
    """
    source_df = spark_session.createDataFrame([], StructType([
        StructField("POLICY_STATE", StringType(), True),
        StructField("BUR", StringType(), True)
    ]))
    lookup_df = spark_session.createDataFrame([], StructType([
        StructField("LKP_ROW_WID", StringType(), True),
        StructField("LKP_INTEGRATION_ID", StringType(), True),
        StructField("LKP_NEW_BUR", StringType(), True)
    ]))
    mock_read_table.side_effect = [lookup_df]
    config = {"lookup_table": "catalog.database.lookup_table"}
    instance = module_under_test(spark_session, config)

    insert_df, update_df = instance.transform(source_df)

    assert insert_df.count() == 0
    assert update_df.count() == 0
    mock_log_df_info.assert_called()

@patch("module_under_test.log_df_info")
@patch("module_under_test.DataFrame.write")
def test_load_valid_data(mock_write, mock_log_df_info, spark_session):
    """
    Test load method with valid data.
    """
    insert_df = spark_session.createDataFrame(
        [("I", "2023-10-01 00:00:00", "W_CLAIM_CD_BUR_SCD3", "BUR1")],
        StructType([
            StructField("o_Flag", StringType(), True),
            StructField("CDM_INSERT_DT", TimestampType(), True),
            StructField("TGT_TABLE_NAME", StringType(), True),
            StructField("BUR", StringType(), True)
        ])
    )
    update_df = spark_session.createDataFrame(
        [("U", "2023-10-01 00:00:00", "W_CLAIM_CD_BUR_SCD3", "BUR2")],
        StructType([
            StructField("o_Flag", StringType(), True),
            StructField("CDM_UPDATE_DT", TimestampType(), True),
            StructField("TGT_TABLE_NAME", StringType(), True),
            StructField("BUR", StringType(), True)
        ])
    )
    config = {"insert_target_path": "/path/to/insert_target"}
    instance = module_under_test(spark_session, config)

    instance.load(insert_df, update_df)

    mock_write.format.assert_called_with("delta")
    mock_write.mode.assert_called_with("append")
    mock_write.save.assert_called_with("/path/to/insert_target")
    mock_log_df_info.assert_called()

# ======================================================================
# Tests for: Chunk 8 (Lines 641-708)
# ======================================================================

def spark_session():
    return SparkSession.builder.master("local").appName("pytest").getOrCreate()

@patch("pyspark.sql.DataFrame.write")
def test_load_insert_data(mock_write, spark_session):
    """
    Test the load method for inserting data into the target table.
    """
    # Create test data
    test_data = [("I", "2023-10-01 12:00:00", "table1", "BUR1")]
    schema = StructType([
        StructField("o_Flag", StringType(), True),
        StructField("CDM_UPDATE_DT", TimestampType(), True),
        StructField("TGT_TABLE_NAME", StringType(), True),
        StructField("BUR", StringType(), True)
    ])
    insert_df = spark_session.createDataFrame(test_data, schema)

    # Mock configuration
    config = {'insert_target_path': '/mock/insert_target'}

    # Mock write operation
    mock_write.format.return_value.mode.return_value.save.return_value = None

    # Execute load method
    pipeline = CDMClaimSCD3Pipeline(spark_session, config)
    pipeline.load(insert_df, spark_session.createDataFrame([], schema))

    # Validate write operation
    mock_write.format.assert_called_once_with("delta")
    mock_write.format.return_value.mode.assert_called_once_with("append")
    mock_write.format.return_value.mode.return_value.save.assert_called_once_with('/mock/insert_target')

@patch("pyspark.sql.DataFrame.write")
def test_load_update_data(mock_write, spark_session):
    """
    Test the load method for updating data into the target table.
    """
    # Create test data
    test_data = [("U", "2023-10-01 12:00:00", "table2", "BUR2")]
    schema = StructType([
        StructField("o_Flag", StringType(), True),
        StructField("CDM_UPDATE_DT", TimestampType(), True),
        StructField("TGT_TABLE_NAME", StringType(), True),
        StructField("BUR", StringType(), True)
    ])
    update_df = spark_session.createDataFrame(test_data, schema)

    # Mock configuration
    config = {'update_target_path': '/mock/update_target'}

    # Mock write operation
    mock_write.format.return_value.mode.return_value.save.return_value = None

    # Execute load method
    pipeline = CDMClaimSCD3Pipeline(spark_session, config)
    pipeline.load(spark_session.createDataFrame([], schema), update_df)

    # Validate write operation
    mock_write.format.assert_called_once_with("delta")
    mock_write.format.return_value.mode.assert_called_once_with("append")
    mock_write.format.return_value.mode.return_value.save.assert_called_once_with('/mock/update_target')

@patch("pyspark.sql.DataFrame.write")
def test_load_empty_data(mock_write, spark_session):
    """
    Test the load method with empty DataFrames for both insert and update.
    """
    # Create empty DataFrames
    schema = StructType([
        StructField("o_Flag", StringType(), True),
        StructField("CDM_UPDATE_DT", TimestampType(), True),
        StructField("TGT_TABLE_NAME", StringType(), True),
        StructField("BUR", StringType(), True)
    ])
    empty_df = spark_session.createDataFrame([], schema)

    # Mock configuration
    config = {
        'insert_target_path': '/mock/insert_target',
        'update_target_path': '/mock/update_target'
    }

    # Mock write operation
    mock_write.format.return_value.mode.return_value.save.return_value = None

    # Execute load method
    pipeline = CDMClaimSCD3Pipeline(spark_session, config)
    pipeline.load(empty_df, empty_df)

    # Validate write operation
    mock_write.format.assert_not_called()

@patch("pyspark.sql.DataFrame.write")
def test_load_invalid_data(mock_write, spark_session):
    """
    Test the load method with invalid data (missing required columns).
    """
    # Create invalid DataFrame
    test_data = [("I", "2023-10-01 12:00:00")]
    schema = StructType([
        StructField("o_Flag", StringType(), True),
        StructField("CDM_UPDATE_DT", TimestampType(), True)
    ])
    invalid_df = spark_session.createDataFrame(test_data, schema)

    # Mock configuration
    config = {
        'insert_target_path': '/mock/insert_target',
        'update_target_path': '/mock/update_target'
    }

    # Mock write operation
    mock_write.format.return_value.mode.return_value.save.return_value = None

    # Execute load method
    pipeline = CDMClaimSCD3Pipeline(spark_session, config)
    with pytest.raises(Exception):
        pipeline.load(invalid_df, invalid_df)

@patch("pyspark.sql.DataFrame.write")
def test_load_partial_data(mock_write, spark_session):
    """
    Test the load method with partial data (insert DataFrame is empty, update DataFrame has data).
    """
    # Create test data
    test_data = [("U", "2023-10-01 12:00:00", "table2", "BUR2")]
    schema = StructType([
        StructField("o_Flag", StringType(), True),
        StructField("CDM_UPDATE_DT", TimestampType(), True),
        StructField("TGT_TABLE_NAME", StringType(), True),
        StructField("BUR", StringType(), True)
    ])
    update_df = spark_session.createDataFrame(test_data, schema)
    empty_df = spark_session.createDataFrame([], schema)

    # Mock configuration
    config = {
        'insert_target_path': '/mock/insert_target',
        'update_target_path': '/mock/update_target'
    }

    # Mock write operation
    mock_write.format.return_value.mode.return_value.save.return_value = None

    # Execute load method
    pipeline = CDMClaimSCD3Pipeline(spark_session, config)
    pipeline.load(empty_df, update_df)

    # Validate write operation for update
    mock_write.format.assert_called_once_with("delta")
    mock_write.format.return_value.mode.assert_called_once_with("append")
    mock_write.format.return_value.mode.return_value.save.assert_called_once_with('/mock/update_target')